# Movie Expert Agent

An OpenAI function-calling agent that answers movie questions using the [Nomad Movies API](https://nomad-movies-2.nomadcoders.workers.dev).

**Tools**: `get_popular_movies`, `get_movie_details(id)`, `get_movie_credits(id)`. The LLM picks which to call based on the user's question.

In [1]:
import os

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"

In [2]:
import requests

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").text


def get_movie_details(id):
    return requests.get(f"{BASE_URL}/movies/{id}").text


def get_movie_credits(id):
    return requests.get(f"{BASE_URL}/movies/{id}/credits").text

In [3]:
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [4]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of currently popular movies. Use this when the user asks what movies are popular, trending, or currently in theaters. Takes no arguments.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a single movie by its ID, such as its title, overview, release date and rating. Use this when the user asks about a specific movie identified by an ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie to get details for. e.g. 550",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew (actors, director, etc.) of a movie by its ID. Use this when the user asks who starred in, acted in, or made a specific movie identified by an ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie to get the credits for. e.g. 550",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [ ]:
import openai
import json
from openai.types.chat import ChatCompletionMessage

client = openai.OpenAI()
messages = []


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        # 1. Record the assistant's "I want to call a tool" message
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # 2. Actually run each requested function
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with arguments: {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON arguments: {e}")
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            # 3. Feed the tool result back into the conversation
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        # 4. Let the LLM see the result and produce a final answer (recursion)
        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [6]:
while True:
    message = input("Ask the Movie Expert... (q to quit) ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화가 무엇인지 알려줘
Calling function: get_popular_movies with arguments: {}
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - 개봉일: 2026-05-13
   - 평점: 7.9
   - 개요: 한 남자가 사랑에 빠진 상대의 마음을 얻기 위해 신비로운 '원 위시 윌로우'를 깨뜨리면서 벌어지는 어두운 이야기에 관한 영화.
   ![Obsession](https://image.tmdb.org/t/p/w780/d1AMBt6gxy7Yoz8rHqyCQgvoAXw.jpg)

2. **Peddi**
   - 개봉일: 2026-06-03
   - 평점: 6.1
   - 개요: 1980년대 안드라 프라데시 시골의 한 마을 주민이 스포츠를 통해 커뮤니티를 단결시키는 이야기.
   ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Lee Cronin's The Mummy**
   - 개봉일: 2026-04-15
   - 평점: 8.1
   - 개요: 언론인의 어린 딸이 사막에서 실종된 후 8년 만에 돌아오지만 기이한 사건들이 벌어진다.
   ![Lee Cronin's The Mummy](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

4. **The Mandalorian and Grogu**
   - 개봉일: 2026-05-20
   - 평점: 6.8
   - 개요: 새로운 공화국이 저항군이 싸운 모든 것을 보호하기 위해 전설적인 바운티 헌터의 도움을 받는 이야기.
   ![The Mandalorian and Grogu](https://image.tmdb.org/t/p/w780/5Vi8dSauVwH1HOsiZceDMbRr1Ca.jpg)

5. **Kara**
   - 개봉일: 2026-04-30
   -